# Goodbook Recommendations

In [ ]:
# Install the LibRecommender package
!pip install -U LibRecommender

In [ ]:
# Import packages and configure notebook
import pandas as pd
import libreco.data
import libreco.algorithms 
from libreco.evaluation import evaluate

pd.options.display.max_rows = 99
pd.options.display.max_columns = 99

%matplotlib inline

## Reading the data

This assumes the files are located in "data" folder in the folder where the notebook is

In [ ]:
# Reading the ratings and book files into pandas DataFrames

df_books = pd.read_csv("data/books.csv")
df_ratings = pd.read_csv('data/ratings.csv')

df_ratings.head()

Data is loaded, ready to do some Exploratory Data Analysis...

## Recommendations with LibRecommender package

[LibRecommender Documentation](https://librecommender.readthedocs.io/en/latest/index.html)

LibRecommender expects our columns to be named `user`, `item`, and `label`, so we need to rename the colums in `df_ratings`, [see documentation](https://librecommender.readthedocs.io/en/latest/user_guide/data_processing.html#data-processing)

In [ ]:
df_ratings.rename(
    columns={"user_id": "user", "book_id": "item", "rating": "label"},
    inplace=True
)
df_ratings.head()

The next step is to create a train (and test) set from the data for a recommender model,

e.g. using `libreco.data.random_split` [(see documentation)](https://librecommender.readthedocs.io/en/latest/api/data/split.html#libreco.data.random_split)


For more possibilities see https://librecommender.readthedocs.io/en/latest/api/data/split.html

In [ ]:
# Split the data into a train and test set (20%) randomly
df_train, df_test = libreco.data.random_split(df_ratings, test_size=0.2)
train_data, data_info = libreco.data.DatasetPure.build_trainset(df_train)
test_data = libreco.data.DatasetPure.build_evalset(df_test)

Now you should be ready to try one of the [recommender algorithms](https://librecommender.readthedocs.io/en/latest/api/algorithms/index.html) and [evaluate accuracy](https://librecommender.readthedocs.io/en/latest/user_guide/evaluation_save_load.html). Below you will find a function that implements precision and recall at k.


In [ ]:
def precision_recall_at_k(df_test, k=10, threshold=3.5):
    """Calculate precision and recall at k metrics for each user"""

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in df_test.groupby("user"):

        # Number of relevant items
        n_rel = sum(user_ratings["label"] >= threshold)

        # Sort user ratings by estimated value
        topk = user_ratings.sort_values(by="pred", ascending=False).head(k)

        # Number of recommended items in top k
        n_rec_k = sum((topk["pred"] >= threshold))

        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(
            ((topk["label"] >= threshold) & (topk["pred"] >= threshold))
        )

        # Precision@K: Proportion of recommended items that are relevant
        # When n_rec_k is 0, Precision is undefined. We here set it to 0.
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Recall@K: Proportion of relevant items that are recommended
        # When n_rel is 0, Recall is undefined. We here set it to 0.
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
        
    precision_at_k = sum(precisions.values())/len(precisions)
    recall_at_k = sum(recalls.values())/len(recalls)
    
    print(f'precision at {k}:\t{precision_at_k:0.4f}')
    print(f'recall at {k}.  :\t{recall_at_k:0.4f}')

    return precision_at_k, recall_at_k

#### Basic example of training a model and getting predictions

In [ ]:
# Train an Item-based Nearest neighbourhood model with 20 neighbours and using Cosine similarity as similarity measure
# shows the rmse on the test set
model = libreco.algorithms.ItemCF(data_info=data_info, task="rating", sim_type="cosine", k_sim=20)
model.fit(
    train_data, 
    neg_sampling=False,
    eval_data=test_data,
    metrics=["rmse"],
    verbose=10
)

In [ ]:
# get a prediction of the rating for user_id 1 on book_id 2
predicted_rating = model.predict(user=1, item=2)
predicted_rating

In [ ]:
# predict ratings for the complete test set
df_test["pred"] = model.predict(df_test["user"], df_test["item"])
df_test

In [ ]:
# calculate precision at recall at K over the test set
precision, recall = precision_recall_at_k(df_test)
